In [15]:
import os
import time
import tomllib
import sqlparse
import warnings
import pymysql
import numpy as np
import pandas as pd
import datetime as dt
from tqdm import tqdm
from dotenv import load_dotenv

# Vectorization vs Parallelization

For June 2026, the seismic revision routine makes a search of 17 different checks on the seismic data. Each check is a function that takes in the seismic data and performs some operations on it to check for certain conditions. The checks are performed sequentially, which means that each check is performed one after the other. This can be time-consuming, especially if the seismic data is large. In order to handle this, initially parallelization was implemented using the multiprocessing library in Python. This allowed the checks to be performed in parallel, which significantly reduced the time taken to perform the checks. However, this approach had some limitations, such as the overhead of creating and managing multiple processes, and the need to ensure that the checks were thread-safe.

Furthermore, there are some operations that are performed on the seismic data that can be vectorized using libraries such as NumPy. Vectorization allows for the operations to be performed on entire arrays of data at once, rather than iterating through each element individually. This can significantly reduce the time taken to perform the operations, as it takes advantage of the underlying hardware optimizations for array operations. In addition, there is no need to initialize multiple workers or manage the overhead associated with parallelization. By using vectorization, we can reduce energy consumption and improve the efficiency of the seismic revision routine, while also simplifying the code and reducing the potential for errors. Overall, while parallelization can be useful in certain situations, we will check if vectorization is a more efficient and effective approach for handling large datasets and performing complex operations on them.

In this notebook, we will compare the performance of vectorization and parallelization for a specific check in the seismic revision routine. We will implement both approaches and measure the time taken to perform the check on a sample seismic dataset. We will also analyze the results and discuss the advantages and disadvantages of each approach. Finally, we will make recommendations on which approach to use for different scenarios in the seismic revision routine. The main idea is to define single functions to perform each check, and then use either vectorization or parallelization to apply those functions to the seismic data.

## Query seismic data from database

Let's start by querying the seismic data from the database. We will use the `pymysql` library to connect to the database and execute a SQL query to retrieve the seismic data. We will also use the `dotenv` library to load the database credentials from a `.env` file, as stated in the previous notebook.

In [2]:
env_path = os.path.join(os.getcwd(), '.env')
load_dotenv(dotenv_path=env_path)

def connect_to_db(
        query: str,
        start_time: dt.datetime = None,
        end_time: dt.datetime = None,
        **kwargs):

    if start_time and end_time:
        start_time_str = start_time.strftime("%Y-%m-%d %H:%M:%S")
        end_time_str = end_time.strftime("%Y-%m-%d %H:%M:%S")
        full_query = f"{query} '{start_time_str}' and '{end_time_str}' ORDER BY Origin.time_value ASC;"  # Filter and order by time_value
    else:
        full_query = query

    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        db_connection = pymysql.connect(
            host=os.getenv('SERVER_HOST'),
            user=os.getenv('SERVER_USERNAME'),
            password=os.getenv('SERVER_PASSWORD'),
            db=os.getenv('SERVER_DATABASE')
        )

        try:
            with tqdm(total=1, desc='Querying database...', unit='query', leave=False,bar_format="{desc}") as pbar:
                df = pd.read_sql_query(full_query, db_connection, **kwargs)
                pbar.update(1)
        finally:
            db_connection.close()

    return df

In [3]:
# Read revision.sql file
with open('./queries/revision.sql', 'r') as file:
    revision_query = file.read()

clean_sql = sqlparse.format(revision_query, strip_comments=True).strip()

initial_time = dt.datetime(2023, 5, 3, 0, 0, 0)
final_time = dt.datetime(2026, 3, 17, 0, 0, 0)
event_df3 = connect_to_db(clean_sql, start_time=initial_time, end_time=final_time)

print(f"Number of rows in the seismic data: {len(event_df3)}")
# Print how events are by event_type
print("Number of events by event_type:")
print(event_df3['event_type'].value_counts())

Number of rows in the seismic data: 209243
Number of events by event_type:
event_type
not locatable                  116633
earthquake                      78301
not existing                     5211
explosion                        4372
outside of network interest      4200
volcanic eruption                 455
induced earthquake                  3
Name: count, dtype: int64


## 1. Comparison checks

The seismic revision routine performs a list of quality checks on earthquakes, based on relational or absolute thresholds. These checks are designed to identify earthquakes that may not be reliable and may require further investigation. Some of the checks that are performed include:

1. High RMS: This check identifies earthquakes with a high root-mean-square (RMS) value, which indicates that the seismic data is noisy and may not be reliable. The threshold for this check is typically set at a certain value, such as 1.51.
2. Localization uncertainty: This check identifies earthquakes with a high localization uncertainty, which indicates that the location of the earthquake is not well-defined. The threshold for this check is typically set at a certain value, such as 12 km. It is applied both on latitude, longitude, and depth.
3. Depth check: This check identifies earthquakes with a depth that is outside of a certain range, such as between 0 and 700 km. This check is important because earthquakes that are too shallow or too deep may not be reliable and may require further investigation.

For all these type of checks, it is possible to vectorize the solution by applying the check to the entire dataset at once, rather than iterating through each earthquake individually. The idea here is to create a single general function, receiving the filtered seismic data, the threshold value or values (if there are multiple thresholds), and the column to be checked. The function will then apply the check to the entire dataset and return a boolean mask indicating which earthquakes meet the criteria for being flagged as unreliable. This approach can significantly reduce the time taken to perform the checks, without making it too complicated to be debugged or maintained.

In [4]:
# Previous version
def single_check(event):
    observations = []

    # First check: High RMS values
    exceptions = ["not locatable", "outside of network interest", "volcanic eruption", "explosion", "not existing"]
    if event['quality_standardError'] > 1.51 and event['event_type'] not in exceptions:
        observations.append("High RMS value")

    if len(observations) > 0:  # If the event has observations, return the information
        return event, observations
    else:
        return None, None

# For loop version
time1 = time.time()
results = []
for _, event in event_df3.iterrows():
    result, obs = single_check(event)
    if result is not None:
        results.append((result, obs))
high_rms_df_loop = pd.DataFrame([res[0] for res in results])
time2 = time.time()
print(f"Number of events with high RMS (loop version): {len(high_rms_df_loop)}")
print(f"Time taken for high RMS check (loop version): {time2 - time1:.4f} seconds")

Number of events with high RMS (loop version): 16
Time taken for high RMS check (loop version): 12.4594 seconds


In [5]:
# Vectorized function to make comparison between a column and a threshold value
def build_quality_mask(
    events: pd.DataFrame,
    column: str,
    mode: str,
    threshold=None,
    lower=None,
    upper=None,
    dtype=np.float64
) -> np.ndarray:
    """
    Vectorized generic comparator for seismic quality checks.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    column : str
        Column to evaluate.
    mode : str
        Comparison mode:
            'gt'       -> values > threshold
            'ge'       -> values >= threshold
            'lt'       -> values < threshold
            'le'       -> values <= threshold
            'between'  -> lower <= values <= upper
            'outside'  -> values < lower or values > upper
            'abs_gt'   -> abs(values) > threshold
            'abs_ge'   -> abs(values) >= threshold
    threshold : float, optional
        Threshold for one-sided comparisons.
    lower, upper : float, optional
        Bounds for range comparisons.
    dtype : numpy dtype
        Target dtype for NumPy conversion.

    Returns
    -------
    np.ndarray
        Boolean mask of flagged rows.
    """
    values = events[column].to_numpy(dtype=dtype, copy=False)

    if mode == 'gt':
        return values > threshold
    elif mode == 'ge':
        return values >= threshold
    elif mode == 'lt':
        return values < threshold
    elif mode == 'le':
        return values <= threshold
    elif mode == 'between':
        return (values >= lower) & (values <= upper)
    elif mode == 'outside':
        return (values < lower) | (values > upper)
    elif mode == 'abs_gt':
        return np.abs(values) > threshold
    elif mode == 'abs_ge':
        return np.abs(values) >= threshold
    else:
        raise ValueError(f"Unsupported mode: {mode}")

# Check for high RMS values on 'earthquake' and 'volcanic eruption' event types
time1 = time.time()
rms_threshold = 1.51
rms_mask = build_quality_mask(
    events=event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])],
    column='quality_standardError',
    mode='gt',
    threshold=rms_threshold
)
high_rms_df = event_df3[event_df3['event_type'].isin(['earthquake', 'volcanic eruption'])][rms_mask]
time2 = time.time()
print(f"Number of events with high RMS: {len(high_rms_df)}")
print(f"Time taken for high RMS check: {time2 - time1:.4f} seconds")

Number of events with high RMS: 16
Time taken for high RMS check: 0.0955 seconds


As you can see, the vectorized version of the high RMS check is significantly faster than the loop version (13.53 s to just 0.0812 s!). This is because the vectorized version takes advantage of NumPy's optimized array operations, which are implemented in C and can be executed much faster than Python loops. In contrast, the loop version iterates through each event one by one, which is much slower, especially for large datasets. Additionally, the vectorized version is more concise and easier to read, as it eliminates the need for explicit loops and conditional statements. Overall, this demonstrates the significant performance benefits of using vectorization for data processing tasks in Python.

Now let's create a wrapper function to apply multiple checks at once:

In [22]:
# Wrapper function to apply multiple checks at once
def seismic_quality_checks(events: pd.DataFrame) -> pd.DataFrame:
    """
    Apply common earthquake quality checks and return flagged events.
    """
    selections = events[events['event_type'].eq('earthquake')].reset_index(drop=True)

    if selections.empty:
        return selections.iloc[0:0].copy()

    masks = {
        'High RMS': build_quality_mask(
            selections, column='quality_standardError', mode='gt', threshold=1.51
        ),
        'High err_lat': build_quality_mask(
             selections, column='latitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_lon': build_quality_mask(
            selections, column='longitude_uncertainty', mode='gt', threshold=12.0
        ),
        'High err_depth': build_quality_mask(
            selections, column='depth_uncertainty', mode='gt', threshold=12.0
        ),
        'Invalid depth': build_quality_mask(
            selections, column='depth_value', mode='outside', lower=0.0, upper=200.0
        ),
        'Earthquake with 6 or less phase count' : build_quality_mask(
            selections, column='quality_associatedPhaseCount', mode='le', threshold=6.0
        )
    }

    combined_mask = np.zeros(len(selections), dtype=bool)
    for mask in masks.values():
        combined_mask |= mask

    flagged = selections.loc[combined_mask].copy()

    flagged_idx = np.where(combined_mask)[0]
    observations = []
    for i in flagged_idx:
        obs = [name for name, mask in masks.items() if mask[i]]
        observations.append(', '.join(obs))

    flagged['Observations'] = observations
    return flagged.reset_index(drop=True)

# Apply the checks and measure time
time1 = time.time()
flagged_events = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged_events)}")
print(f"Time taken for seismic quality checks: {time2 - time1:.4f} seconds")

Number of flagged events: 505
Time taken for seismic quality checks: 0.0885 seconds


The wrapper function `seismic_quality_checks` applies multiple quality checks to the seismic data and returns a DataFrame of flagged events along with the observations for each event. The function first filters the input DataFrame to include only earthquake events, and then applies each check using the `build_quality_mask` function. The results of all checks are combined into a single boolean mask, which is used to select the flagged events. Finally, the observations for each flagged event are compiled into a new column in the resulting DataFrame.

Now, it is time to add more checks to the wrapper function in order to make it more comprehensive. To maintain the readability and simplicity of the code, we will use a TOML file to store the configuration for each check, including the column and event types to be checked, the mode of comparison, and the threshold values. This way, we can easily add or modify checks without having to change the code of the wrapper function itself. The wrapper function will read the configuration from the TOML file and apply the checks accordingly. This approach allows us to keep the code clean and maintainable while still providing a flexible way to manage the quality checks for seismic events.

In [16]:
# Read TOML file without comments
with open('./seismic_checks.toml', 'rb') as f:
    checks_config = tomllib.load(f)["checks"]

checks_config

[{'name': 'High RMS',
  'column': 'quality_standardError',
  'mode': 'ge',
  'threshold': 1.5,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Latitude Uncertainty',
  'column': 'latitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Longitude Uncertainty',
  'column': 'longitude_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'High Depth Uncertainty',
  'column': 'depth_uncertainty',
  'mode': 'gt',
  'threshold': 12,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'Negative Depth',
  'column': 'depth_value',
  'mode': 'lt',
  'threshold': 0,
  'event_type': ['all']},
 {'name': 'Noncommon High Depth',
  'column': 'depth_value',
  'mode': 'ge',
  'threshold': 200,
  'event_type': ['earthquake', 'explosion', 'volcanic eruption']},
 {'name': 'Event with 7 or less 

In [50]:
def load_checks(path: str = "seismic_checks.toml") -> list[dict]:
    """Load quality checks config from a TOML file."""
    with open(path, "rb") as f:
        return tomllib.load(f)["checks"]


def seismic_quality_checks(
    events: pd.DataFrame,
    checks_path: str = "seismic_checks.toml",
) -> pd.DataFrame:
    """
    Apply seismic quality checks loaded from a TOML config file.

    Each check specifies its own target event_type list, so different checks
    can apply to different subsets of the dataset. The special keyword "all"
    means the check applies to every event type present in the data.

    Parameters
    ----------
    events : pd.DataFrame
        Input seismic dataframe.
    checks_path : str
        Path to the TOML config file.

    Returns
    -------
    pd.DataFrame
        Flagged events with an 'Observations' column listing all triggered
        checks per event. Each publicID appears at most once.
    """
    checks = load_checks(checks_path)

    if events.empty:
        return events.iloc[0:0].copy()

    # observations_map: { row_index -> [check_name, ...] }
    observations_map: dict[int, list[str]] = {}

    for check in checks:
        # Select only rows matching this check's event types
        subset = events[events["event_type"].isin(check["event_type"])]

        if subset.empty:
            continue

        # Build the quality mask on the subset using original index
        kwargs = {k: v for k, v in check.items() if k not in ("name", "event_type")}
        flagged_mask = build_quality_mask(subset, **kwargs)

        # Map triggered rows back to original DataFrame index
        flagged_original_idx = subset.index[flagged_mask]
        for idx in flagged_original_idx:
            observations_map.setdefault(idx, []).append(check["name"])

    if not observations_map:
        return events.iloc[0:0].copy()

    # Build output from all flagged original indices — each event appears once
    flagged_idx = sorted(observations_map.keys())
    flagged = events.loc[flagged_idx].copy()
    flagged["Observations"] = [
        ", ".join(observations_map[i]) for i in flagged_idx
    ]

    return flagged.reset_index(drop=True)

In [51]:
time1 = time.time()
flagged = seismic_quality_checks(event_df3)
time2 = time.time()
print(f"Number of flagged events: {len(flagged)}")
print(f"Time taken for seismic quality checks with TOML config: {time2 - time1:.4f} seconds")

Number of flagged events: 2739
Time taken for seismic quality checks with TOML config: 0.4214 seconds


In [28]:
flagged

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment,Observations
0,2023-05-03 00:22:16,SGC2023ipzion,-4.941406,0.885335,0.445758,5.79034,2.080092,5.282553,8.0,8.0,...,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.781972,-77.914268,MLr,NonLinLoc,Poveda_et_al_2018,None,Negative Depth
1,2023-05-03 05:43:15,SGC2023iqjzej,-0.280000,NaN,0.190000,8.60000,1.838478,1.838478,7.0,7.0,...,not locatable,SGC,"Colombia-Ecuador, Region Fronteriza",0.772333,-77.915833,None,Hypo71,RSNC,None,Negative Depth
2,2023-05-04 14:46:31,SGC2023isxsdy,32.960000,1.085318,1.680000,10.50000,5.444722,5.444722,13.0,13.0,...,earthquake,SGC,"Zapatoca - Santander, Colombia",6.828333,-73.292667,MLr_vmm,Hypo71,VMM,None,High RMS
3,2023-05-04 17:33:31,SGC2023itdgcn,3.000000,1.787111,1.900000,9.40000,9.970206,9.970206,9.0,9.0,...,explosion,SGC,"El Paso - Cesar, Colombia",9.627000,-73.477667,MLr_4,Hypo71,modelCesar2,None,High RMS
4,2023-05-04 17:48:15,SGC2023itdsum,10.000000,NaN,2.499988,NaN,NaN,NaN,NaN,NaN,...,explosion,SGC,"Barrancas - la Guajira, Colombia",11.020000,-72.881600,None,,,None,High RMS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2998,2026-03-15 17:30:01,SGC2026ffivca,-0.450000,2.005201,0.940000,39.20000,5.868986,5.868986,11.0,11.0,...,explosion,SGC,"El Paso - Cesar, Colombia",9.600333,-73.521167,MLr_4,Hypo71,modelCesar2,None,"High Depth Uncertainty, Negative Depth"
2999,2026-03-15 17:55:58,SGC2026ffjrlg,10.000000,1.339524,12.757219,NaN,NaN,NaN,NaN,NaN,...,explosion,SGC,"Barrancas - La Guajira, Colombia",11.020000,-72.881600,MLr_4,,,None,High RMS
3000,2026-03-15 17:56:26,SGC2026ffjrvs,0.000000,1.954542,0.860000,84.30000,39.103005,39.103005,7.0,7.0,...,explosion,SGC,"Riohacha - La Guajira, Colombia",11.179000,-72.751333,MLr_4,Hypo71,RSNC,None,"High Latitude Uncertainty, High Longitude Unce..."
3001,2026-03-16 00:46:39,SGC2026ffxhio,0.000000,4.186005,0.801828,0.00000,17.968908,8.162779,9.0,9.0,...,earthquake,SGC,OcÃ©ano PacÃ­fico,4.017380,-82.418900,mb,LOCSAT,iasp91,None,High Latitude Uncertainty


In [30]:
# Filter events from 2026-03-01 to 2026-03-17
march_events = event_df3[
    (event_df3['time_value'] >= dt.datetime(2026, 3, 1)) &
    (event_df3['time_value'] <= dt.datetime(2026, 3, 17))
].copy()
march_events

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
205759,2026-03-01 00:09:53,SGC2026eeieln,10.000000,0.683220,2.478282,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"MurindÃ³ - Antioquia, Colombia",6.989500,-76.697100,MLr_1,,,None
205760,2026-03-01 00:18:00,SGC2026eeilng,5.000000,1.743837,0.090000,NaN,NaN,NaN,5.0,5.0,...,NaN,not locatable,SGC,"Riosucio - ChocÃ³, Colombia",7.213833,-77.126000,MLr_1,Hypo71,RSNC,None
205761,2026-03-01 00:19:42,SGC2026eeimxl,10.000000,-0.088892,3.748619,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
205762,2026-03-01 00:22:47,SGC2026eeippb,12.226562,1.045598,0.253937,4.556525,7.031040,2.865880,8.0,8.0,...,NaN,earthquake,SGC,"Dabeiba - Antioquia, Colombia",6.991690,-76.253300,MLr_1,NonLinLoc,Poveda_et_al_2018,None
205763,2026-03-01 00:23:23,SGC2026eeiqbz,10.000000,0.033025,1.093982,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209238,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,27.0,...,NaN,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209239,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209240,2026-03-16 23:31:36,SGC2026fhqnnx,10.000000,-0.143127,13.207386,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None
209241,2026-03-16 23:35:30,SGC2026fhqqxh,10.000000,-0.012095,223.843471,NaN,NaN,NaN,NaN,NaN,...,NaN,not locatable,SGC,"Dabeiba - Antioquia, Colombia",7.018000,-76.210000,MLr_1,,,None


In [52]:
# Make checks for those events and see how many are flagged
time1 = time.time()
flagged_march = seismic_quality_checks(march_events)
time2 = time.time()
print(f"Number of flagged events in March 2026: {len(flagged_march)}")
print(f"Time taken for seismic quality checks on March 2026 events: {time2 - time1:.4f} seconds")

Number of flagged events in March 2026: 14
Time taken for seismic quality checks on March 2026 events: 0.0308 seconds


In [48]:
flagged_march[['time_value', 'publicID', 'event_type', 'text', 'Observations']]

,time_value,publicID,event_type,text,Observations
0,2026-03-07 03:13:02,SGC2026epopop,not locatable,"Tesalia - Huila, Colombia",Negative Depth
1,2026-03-11 17:28:59,SGC2026exzwrg,explosion,"La Jagua de Ibirico - Cesar, Colombia","High Latitude Uncertainty, High Longitude Unce..."
2,2026-03-12 17:30:29,SGC2026ezvqvs,explosion,"AgustÃ­n Codazzi - Cesar, Colombia",High RMS
3,2026-03-12 17:39:12,SGC2026ezvyjf,not locatable,"Murillo - Tolima, Colombia",Negative Depth
4,2026-03-12 19:04:54,SGC2026ezyufi,explosion,"Riohacha - La Guajira, Colombia","High RMS, High Latitude Uncertainty, High Long..."
5,2026-03-12 19:05:31,SGC2026ezyusv,explosion,"Barrancas - La Guajira, Colombia",High RMS
6,2026-03-13 18:08:51,SGC2026fbsqud,explosion,"Barrancas - La Guajira, Colombia",High RMS
7,2026-03-14 17:27:46,SGC2026fdnagp,explosion,"Becerrill - Cesar, Colombia",High RMS
8,2026-03-14 17:29:55,SGC2026fdncda,explosion,"ChiriguanÃ¡ - Cesar, Colombia",High RMS
9,2026-03-15 17:30:01,SGC2026ffivca,explosion,"El Paso - Cesar, Colombia",Negative Depth


In [54]:
subset = event_df3[event_df3["event_type"].isin(['earthquake'])]   # Check earthquakes with RMS = 0.0? ASK
subset

,time_value,publicID,depth_value,magnitude_value,quality_standardError,depth_uncertainty,latitude_uncertainty,longitude_uncertainty,quality_associatedPhaseCount,quality_usedPhaseCount,...,quality_associatedStationCount,event_type,creationInfo_agencyID,text,latitude_value,longitude_value,magnitude_type,methodID,earthModelID,comment
2,2023-05-03 01:17:28,SGC2023iqbedm,135.859375,1.911117,0.959775,9.021321,6.419839,7.434003,19.0,19.0,...,NaN,earthquake,SGC,"Zapatoca - Santander, Colombia",6.766267,-73.218474,MLr_3,NonLinLoc,Poveda_et_al_2018,None
3,2023-05-03 01:24:17,SGC2023iqbkaq,22.539062,0.880860,0.356193,7.230936,5.458100,3.798461,12.0,12.0,...,NaN,earthquake,SGC,"PurificaciÃ³n - Tolima, Colombia",3.844233,-74.806306,MLr_2,NonLinLoc,Poveda_et_al_2018,None
8,2023-05-03 02:30:41,SGC2023iqdpgd,13.046875,1.235740,0.658681,9.232364,2.653403,2.414232,11.0,11.0,...,NaN,earthquake,SGC,"Pueblo Rico - Risaralda, Colombia",5.246513,-76.113500,MLr_1,NonLinLoc,Poveda_et_al_2018,None
9,2023-05-03 02:41:19,SGC2023iqdykg,10.000000,1.899968,1.115665,0.000000,4.456467,2.795953,8.0,7.0,...,5.0,earthquake,SGC,Panama,7.278529,-78.895035,MLr,LOCSAT,iasp91,None
15,2023-05-03 03:57:37,SGC2023iqgmef,22.070312,1.102264,0.706856,11.369056,3.366359,3.571194,14.0,13.0,...,NaN,earthquake,SGC,"San Vicente del CaguÃ¡n - CaquetÃ¡, Colombia",2.874808,-74.838579,MLr_2,NonLinLoc,Poveda_et_al_2018,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
209229,2026-03-16 21:36:45,SGC2026fhmspg,15.976562,1.744425,0.647517,6.591403,3.032241,3.268659,17.0,17.0,...,NaN,earthquake,SGC,"Dabeiba - Antioquia, Colombia",6.835058,-76.295056,MLr_1,NonLinLoc,Poveda_et_al_2018,None
209232,2026-03-16 21:48:20,SGC2026fhncoj,93.920000,2.003283,0.180000,1.600000,1.414214,1.414214,13.0,13.0,...,NaN,earthquake,SGC,"Toro - Valle del Cauca, Colombia",4.585000,-76.153833,MLr_1,Hypo71,RSNC,None
209238,2026-03-16 23:11:29,SGC2026fhpwfl,136.328125,2.074670,0.956817,7.253506,5.312920,6.076712,28.0,27.0,...,NaN,earthquake,SGC,"CÃ¡chira - Norte de Santander, Colombia",7.756859,-73.141695,MLr_3,NonLinLoc,Poveda_et_al_2018,None
209239,2026-03-16 23:18:05,SGC2026fhqbxv,103.632812,1.727774,0.597553,8.010460,4.339104,4.863137,22.0,22.0,...,NaN,earthquake,SGC,"Pauna - BoyacÃ¡, Colombia",5.615868,-73.956240,MLr_3,NonLinLoc,Poveda_et_al_2018,None


### Appendix: Locatable events

A check of the seismic revision routine is to identify events that are locatable, but are labeled incorrectly as "not locatable". At RSNC, a event is locatable if it has at least 4 p phases and 2 s phases associated to it. This check is important because it can help to identify earthquakes that may have been misclassified and may require further investigation.

However, the previous routine only checks a verification on the _'quality_associatedPhaseCount'_ column to be greater than or equal to 8 (plus one due to an event with 6 phases in seiscomp will have a _quality_associatedPhaseCount_ of 7). Then, an event with for example 8 p phases and 0 s phases would be flagged as locatable, which is not correct.

The challenge here is that query the number of p and s phases associated to each event requires a join between the _Origin_ and _Arrival_ tables in the database, which can be time-consuming, especially for large datasets. Additionally, the check needs to be performed for each event individually, which can further increase the time taken to perform the check. Therefore, the strategy here is to use the columns _quality_associatedPhaseCount_, _quality_usedPhaseCount_, _quality_usedStationCount_ and _quality_associatedStationCount_ to create a vectorized check that can identify locatable events without the need for a join between the tables. This approach can significantly reduce the time taken to perform the check, while still providing accurate results.

In [7]:
# First example: 3 p and 3 s event (within 2026-03-01 11:13:00 and 2026-03-01 11:14:00)
start_filter = dt.datetime(2026, 3, 1, 11, 13, 0)
end_filter = dt.datetime(2026, 3, 1, 11, 14, 0)
subset_df = event_df3[(event_df3['time_value'] >= start_filter) & (event_df3['time_value'] <= end_filter)].copy()
subset_df[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205886,2026-03-01 11:13:54,not locatable,7.0,7.0,6.0,NaN


In [8]:
# Second example: 4 p and 3 s event (within 2026-03-01 06:38:00 and 2026-03-01 06:39:00)
start_filter_2 = dt.datetime(2026, 3, 1, 6, 38, 0)
end_filter_2 = dt.datetime(2026, 3, 1, 6, 39, 0)
subset_df_2 = event_df3[(event_df3['time_value'] >= start_filter_2) & (event_df3['time_value'] <= end_filter_2)].copy()
subset_df_2[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205831,2026-03-01 06:38:24,earthquake,8.0,8.0,7.0,NaN


In [9]:
# Third example: 4 p and 4 s event (within 2026-03-01 00:22:00 and 2026-03-01 00:23:00)
start_filter_3 = dt.datetime(2026, 3, 1, 0, 22, 0)
end_filter_3 = dt.datetime(2026, 3, 1, 0, 23, 0)
subset_df_3 = event_df3[(event_df3['time_value'] >= start_filter_3) & (event_df3['time_value'] <= end_filter_3)].copy()
subset_df_3[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205762,2026-03-01 00:22:47,earthquake,8.0,8.0,4.0,NaN


In [10]:
# Fourth example: 4 p and 4 s event (within 2026-03-01 11:33:00 and 2026-03-01 11:34:00)
start_filter_4 = dt.datetime(2026, 3, 1, 11, 33, 0)
end_filter_4 = dt.datetime(2026, 3, 1, 11, 34, 0)
subset_df_4 = event_df3[(event_df3['time_value'] >= start_filter_4) & (event_df3['time_value'] <= end_filter_4)].copy()
subset_df_4[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205889,2026-03-01 11:33:07,earthquake,9.0,9.0,8.0,NaN


In [11]:
# Fifth example: 44 p (43 used) and 40 s (81 used picks to locate and 84 total picks) event (within 2026-03-01 09:37:00 and 2026-03-01 09:38:00)
start_filter_5 = dt.datetime(2026, 3, 1, 9, 37, 0)
end_filter_5 = dt.datetime(2026, 3, 1, 9, 38, 0)
subset_df_5 = event_df3[(event_df3['time_value'] >= start_filter_5) & (event_df3['time_value'] <= end_filter_5)].copy()
subset_df_5[['time_value', 'event_type', 'quality_associatedPhaseCount', 'quality_usedPhaseCount', 'quality_usedStationCount', 'quality_associatedStationCount']]

,time_value,event_type,quality_associatedPhaseCount,quality_usedPhaseCount,quality_usedStationCount,quality_associatedStationCount
205864,2026-03-01 09:37:14,earthquake,84.0,81.0,43.0,NaN


In [12]:
# print columns of the dataframe
print(event_df3.columns)

Index(['time_value', 'publicID', 'depth_value', 'magnitude_value',
       'quality_standardError', 'depth_uncertainty', 'latitude_uncertainty',
       'longitude_uncertainty', 'quality_associatedPhaseCount',
       'quality_usedPhaseCount', 'creationInfo_author',
       'quality_usedStationCount', 'quality_associatedStationCount',
       'event_type', 'creationInfo_agencyID', 'text', 'latitude_value',
       'longitude_value', 'magnitude_type', 'methodID', 'earthModelID',
       'comment'],
      dtype='object')
